# 2.7 · 多重比较 / Multiple Testing

> **课程定位**
> 2.6 末尾埋的雷在这里引爆：**检验跑得越多，假阳性越必然**。本课讲两套修正哲学（FWER vs FDR）+ 它们的工业应用（A/B 平台的护栏指标、基因组学）。
> The more tests you run, the more false positives you guarantee. Two correction philosophies and their industrial homes.

> 💡 **面试相关**
> - "跑 20 个指标有 1 个显著，能信吗" ★★★★★（A/B 平台必考）
> - "Bonferroni vs BH 的区别" ★★★★
> - "什么是 p-hacking" ★★★★
> - "FWER vs FDR 各控什么" ★★★

---

## 目录
1. [问题有多严重：假阳性的必然性 ⭐](#1)
2. [绿色软糖：p-hacking 的标准笑话](#2)
3. [FWER 路线：Bonferroni 与 Holm](#3)
4. [FDR 路线：Benjamini-Hochberg ⭐](#4)
5. [模拟对照：三种方法的功效账单](#5)
6. [实战：模拟基因组筛选](#6)
7. [工业实践：A/B 平台怎么处理](#7)
8. [小结](#8)


<a id="1"></a>
## 1. 问题有多严重 / How Bad Is It

$m$ 个**真无效应**的检验全用 $\alpha = 0.05$：

$$\Pr(\text{至少 1 个假阳}) = 1 - (1 - 0.05)^m$$

| m | 至少一个假阳的概率 |
|---|---|
| 1 | 5% |
| 6 | 26%（2.6 节的 Tips 套餐！）|
| 20 | 64% |
| 100 | 99.4% |

**20 个指标的 A/B 测试看板，没效应也几乎保证"发现"点什么。**
A 20-metric dashboard "finds" something nearly every time, even with zero true effects.


In [ ]:
import numpy as np
import scipy.stats as st
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

# 模拟: 一个"无效"改动 × 20 个指标 × 1000 次实验
# A null A/B test watched through 20 metrics, 1000 times
n_exp, m = 1000, 20
at_least_one = 0
for _ in range(n_exp):
    pvals = [st.ttest_ind(rng.normal(0,1,200), rng.normal(0,1,200)).pvalue for _ in range(m)]
    at_least_one += min(pvals) < 0.05

print(f"真效应 = 0, 看 {m} 个指标:")
print(f"  '至少一个显著' 的实验占 {at_least_one/n_exp:.1%}")
print(f"  理论值 1-(0.95)^20 = {1-0.95**20:.1%}")


<a id="2"></a>
## 2. 绿色软糖 / The Green Jelly Bean

[xkcd #882](https://xkcd.com/882/)：科学家检验"软糖致痘"——不显著。按颜色分 20 组再测，**绿色软糖 p<0.05**！次日头条："绿色软糖致痘（95% 置信）"。

这是 **p-hacking 家族**的合影：
| 行为 | 别名 |
|---|---|
| 测 20 个子组报告显著的那个 | subgroup fishing |
| 数据不显著就继续收，显著就停 | **optional stopping** ⭐（A/B 平台头号杀手，Part 19 再算账）|
| 试多种模型/变换/剔除规则，报告"работающий"的 | garden of forking paths |
| 看完数据再定假设 | HARKing |

**共同machinery**：做了 $m$ 次比较，只报告 1 次——分母被藏起来了。
The common mechanism: m comparisons made, one reported — the denominator hidden.


<a id="3"></a>
## 3. FWER 路线：Bonferroni 与 Holm / The FWER Route

**FWER**（family-wise error rate）= $\Pr(\ge 1$ 个假阳$)$。控制它 = "**一个都不许错**"。

**Bonferroni**：每个检验用 $\alpha/m$。
依据 union bound：$\Pr(\bigcup A_i) \le \sum \Pr(A_i) = m \cdot \alpha/m = \alpha$。**无条件有效**（任意相关结构），但 $m$ 大时近乎自杀（$m=10^6$ 基因 → 单测 α = 5e-8）。

**Holm（递进式，永远 ≥ Bonferroni 的功效，同样无条件有效）**：
1. p 值升序排列
2. 第 $i$ 小的和 $\frac{\alpha}{m - i + 1}$ 比
3. 第一次"不显著"出现时停止，后面全不拒绝

**没有理由用 Bonferroni 而不用 Holm**——Holm 严格更强还同样安全（但 Bonferroni 心算方便，口头讨论里活着）。
Holm dominates Bonferroni at zero cost — Bonferroni survives only because it's mental-math friendly.


<a id="4"></a>
## 4. FDR 路线：Benjamini-Hochberg ⭐ / The FDR Route

视角转换：筛 1 万个基因时，"一个都不许错"太奢侈。改控 **FDR** = 期望的"**发现中假阳性的占比**"：

$$\mathrm{FDR} = \mathbb{E}\Big[\frac{\#\text{假发现}}{\#\text{总发现}}\Big] \le q \;(\text{如 } 0.1)$$

"我报 100 个候选基因，**平均最多 10 个是错的**"——后续实验能接受的契约。

**BH 程序**：
1. p 值升序：$p_{(1)} \le \dots \le p_{(m)}$
2. 找**最大**的 $k$ 使 $p_{(k)} \le \frac{k}{m}\,q$
3. 拒绝前 $k$ 个

几何直觉：在 p 值排序图上画斜率 $q/m$ 的直线，**取最后一个落在线下的点**。
Geometric: draw the line with slope q/m on the sorted p-value plot; reject everything up to the last point under it.


In [ ]:
# BH 的几何直觉可视化 / Visualizing the BH line
m = 100
# 90 个真 null + 10 个真效应 / 90 nulls + 10 real effects
p_null = rng.uniform(0, 1, 90)
p_real = rng.beta(0.5, 25, 10)               # 真效应 → p 堆积在 0 附近
pvals = np.concatenate([p_null, p_real])
is_real = np.array([False]*90 + [True]*10)

order = np.argsort(pvals)
p_sorted, real_sorted = pvals[order], is_real[order]
q = 0.10
bh_line = np.arange(1, m+1) / m * q
k = np.max(np.where(p_sorted <= bh_line)[0]) + 1 if np.any(p_sorted <= bh_line) else 0

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(np.arange(1, m+1), p_sorted, s=22,
           c=["red" if r else "gray" for r in real_sorted], alpha=0.8)
ax.plot(np.arange(1, m+1), bh_line, "b-", lw=1.5, label=f"BH line: (i/m)·q, q={q}")
ax.axvline(k, color="green", ls="--", label=f"BH cutoff: reject first k={k}")
ax.axhline(0.05/m, color="purple", ls=":", label=f"Bonferroni α/m = {0.05/m:.4f}")
ax.set_xlim(0, 40); ax.set_ylim(-0.005, 0.15)
ax.set_xlabel("rank i"); ax.set_ylabel("sorted p-value")
ax.legend(); ax.set_title("BH: last point under the line — red = true effects")
plt.tight_layout(); plt.show()

print(f"BH (q=0.1)   拒绝 {k} 个, 其中真效应 {real_sorted[:k].sum()} 个")
print(f"Bonferroni   拒绝 {(p_sorted < 0.05/m).sum()} 个   ← 保守得多")


<a id="5"></a>
## 5. 模拟对照：三种方法的功效账单 / The Power Bill

固定场景（$m=200$，其中 20 个真效应），对比四种策略的 **FWER / FDR / 功效**：


In [ ]:
def simulate_corrections(n_sim=2000, m=200, n_real=20, effect=3.5):
    methods = ["none", "bonferroni", "holm", "fdr_bh"]
    stats = {meth: {"fwer": 0, "fdr": [], "power": []} for meth in methods}
    for _ in range(n_sim):
        # 真效应的 z 检验 p 值 (效应大小 effect), null 的 p ~ U(0,1)
        z_real = rng.normal(effect, 1, n_real)
        p_real = 2 * st.norm.sf(np.abs(z_real))
        p_null = rng.uniform(0, 1, m - n_real)
        pvals = np.concatenate([p_real, p_null])
        truth = np.array([True]*n_real + [False]*(m - n_real))
        for meth in methods:
            if meth == "none":
                rej = pvals < 0.05
            else:
                rej = multipletests(pvals, alpha=0.05 if meth != "fdr_bh" else 0.10,
                                    method=meth)[0]
            fp = (rej & ~truth).sum(); tp = (rej & truth).sum()
            stats[meth]["fwer"] += fp > 0
            stats[meth]["fdr"].append(fp / max(rej.sum(), 1))
            stats[meth]["power"].append(tp / n_real)
    return stats

stats = simulate_corrections()
print(f"m=200 检验, 20 个真效应; α=0.05 (BH 用 q=0.10)")
print(f"{'method':<12} {'FWER':>7} {'FDR':>7} {'power':>7}")
print("-" * 38)
for meth, s in stats.items():
    print(f"{meth:<12} {s['fwer']/2000:>7.1%} {np.mean(s['fdr']):>7.1%} {np.mean(s['power']):>7.1%}")


**账单一目了然**：
- **none**：功效最高但 FWER ≈ 100%——报告里必有假货
- **Bonferroni/Holm**：FWER 压到 5%，**功效掉一大截**——错杀很多真效应
- **BH**：FDR 控制在 ~10%，**功效接近不修正**——"容忍少量假货换大量真货"

**选择哲学**：后果致命（药物批准、上线决策）→ FWER；筛选候选（基因、特征、告警）→ FDR。
Consequence-fatal decisions → FWER; candidate screening → FDR.


<a id="6"></a>
## 6. 实战：模拟基因组筛选 / Simulated Genomics Screen

经典场景：5000 个基因，50 个真差异表达，**找出它们**。


In [ ]:
m, n_real, n_per_group = 5000, 50, 12
truth = np.zeros(m, dtype=bool); truth[:n_real] = True

# 表达矩阵: 病例 vs 对照, 真差异基因 shift 1.8σ
pvals = np.empty(m)
for g in range(m):
    shift = 1.8 if truth[g] else 0.0
    a = rng.normal(0, 1, n_per_group); b = rng.normal(shift, 1, n_per_group)
    pvals[g] = st.ttest_ind(a, b, equal_var=False).pvalue

for name, meth, alpha in [("uncorrected", None, 0.05),
                          ("bonferroni", "bonferroni", 0.05),
                          ("BH (q=0.1)", "fdr_bh", 0.10)]:
    rej = pvals < alpha if meth is None else multipletests(pvals, alpha=alpha, method=meth)[0]
    tp, fp = (rej & truth).sum(), (rej & ~truth).sum()
    print(f"{name:<14}: 报告 {rej.sum():>4} 个基因 | 真 {tp:>2}, 假 {fp:>3} | "
          f"实际 FDP = {fp/max(rej.sum(),1):.1%} | 召回 {tp/n_real:.0%}")


**典型结果**：不修正报告 ~290 个基因但 85% 是假货（后续湿实验全打水漂）；Bonferroni 只敢报个位数（漏掉绝大多数真基因）；**BH 报告几十个、假货占比 ~10%、召回过半**——这正是它统治基因组学的原因。
Uncorrected: ~290 hits, 85% junk. Bonferroni: a handful. BH: dozens of hits, ~10% junk, half the real genes recovered — why it owns genomics.


<a id="7"></a>
## 7. 工业实践：A/B 平台怎么处理 / What A/B Platforms Do

| 实践 | 对应本课概念 |
|---|---|
| **预注册主指标**（1 个，最多 2-3 个）| 把 $m$ 钉死在 1——最强的"修正"是不需要修正 |
| 护栏指标只做单侧"恶化检测" | 减半 α 消耗 |
| 探索性指标打"仅供参考"标签 | 承认未修正，禁止直接决策 |
| 子组分析必须预先声明 | 反 subgroup fishing |
| Sequential testing / always-valid p | 治 optional stopping（Part 19 详述）|
| 多变体实验（A/B/C/D）自动 Dunnett/BH | 平台内建修正 |

> 💡 **面试题"20 个指标 1 个显著怎么办"满分回答**：
> "先问这是不是预注册的主指标。若是探索性的：期望假阳数 = 20×0.05 = 1 个，**这个发现与纯噪声完全一致**。正确动作是把它当作新假设，**在新数据上单独重测**——而不是直接上线。"
> "Expected false positives = 20 x 0.05 = 1 — this 'finding' is exactly consistent with pure noise. Treat it as a new hypothesis and re-test on fresh data."


<a id="8"></a>
## 8. 小结 / Summary

```
m 个检验, 全 α=0.05 → P(≥1 假阳) = 1-0.95^m → m=20 时 64%

两套哲学:
  FWER ("一个都不许错")          FDR ("发现里假货占比 ≤ q")
  ├── Bonferroni: α/m, 无条件     ├── BH: 最大 k 使 p(k) ≤ (k/m)q
  ├── Holm: 递进, 严格优于 Bonf    ├── 功效接近不修正
  └── 用于: 上线/批准类决策        └── 用于: 筛选类任务 (基因/特征/告警)

p-hacking 家族: subgroup fishing / optional stopping / forking paths / HARKing
  └── 共同机制: 藏分母
```

### 💡 面试速查
1. **20 个指标 1 个显著** = 与纯噪声一致 → 新数据重测
2. **Bonferroni**: union bound, 任意相关都有效, 但保守
3. **Holm 严格优于 Bonferroni**, 零代价
4. **BH 控制的是"发现中的假货率"** —— 不是单测错误率
5. **最强修正 = 预注册把 m 钉成 1**

### 下一节
**2.8 功效与样本量**——"要测出 1% 的转化率提升需要多少用户？" 实验设计的算账课。
